# Differentiable convex optimization with `pounce.jax`

`pounce.jax` exposes the convex solve as a **differentiable JAX op**. The
forward pass calls the solver; the backward pass differentiates the
*solution* w.r.t. the problem data by applying the implicit-function theorem
to the KKT system at the optimum (Amos & Kolter, *OptNet*, 2017). This lets
you drop a QP or SOCP inside a larger JAX model and get exact gradients from
`jax.grad` / `jax.jacrev`, and batch with `vmap`/`.batch`.

This notebook builds up:
1. `solve_qp` forward + a gradient, checked against finite differences,
2. the full Jacobian with `jax.jacrev`,
3. gradients w.r.t. the **matrices** $P, G$ — the full OptNet rule,
4. `QpLayer` in a tiny learning loop, and a parallel `.batch`,
5. `solve_socp` — differentiating a second-order cone program.

> `pounce.jax` enables float64 on import (the implicit-diff KKT solve needs
> the precision). Gradients are validated against finite differences in the
> POUNCE test suite; we reproduce a couple of those checks here.

In [1]:
import numpy as np
import jax
import jax.numpy as jnp
from pounce.jax import solve_qp, solve_socp, QpLayer

np.set_printoptions(precision=5, suppress=True)

def fd_grad(f, x, eps=1e-6):
    """Central finite-difference gradient of a scalar f at vector x."""
    x = np.asarray(x, float)
    g = np.zeros_like(x)
    for i in range(x.size):
        e = np.zeros_like(x); e[i] = eps
        g[i] = (f(x + e) - f(x - e)) / (2 * eps)
    return g

## 1. A parametric QP, and its gradient

Equality-constrained QP (smooth in its data — clean for a gradient check):
$$x^\*(c) = \arg\min_x \tfrac12 x^\top P x + c^\top x \quad\text{s.t.}\quad A x = b.$$
Define a scalar loss $\ell(c) = \tfrac12\|x^\*(c) - x_{\text{tgt}}\|^2$ and
compare `jax.grad` to finite differences.

In [2]:
P = jnp.array([[3.0, 0.5], [0.5, 2.0]])
A = jnp.array([[1.0, 1.0]])
b = jnp.array([1.0])
x_tgt = jnp.array([0.3, 0.7])

def x_star(c):
    return solve_qp(P=P, c=c, A=A, b=b)

def loss(c):
    return 0.5 * jnp.sum((x_star(c) - x_tgt) ** 2)

c0 = jnp.array([-1.0, 0.5])
print("x*(c0) :", np.asarray(x_star(c0)))
print("loss   :", float(loss(c0)))

g_ad = np.asarray(jax.grad(loss)(c0))
g_fd = fd_grad(lambda c: float(loss(jnp.asarray(c))), np.asarray(c0))
print("grad (implicit diff):", g_ad)
print("grad (finite diff)  :", g_fd)
assert np.allclose(g_ad, g_fd, atol=1e-5)

x*(c0) : [0.75 0.25]
loss   : 0.2025000000768828


grad (implicit diff): [-0.225  0.225]
grad (finite diff)  : [-0.225  0.225]


## 2. The full solution Jacobian with `jax.jacrev`

$\partial x^\*/\partial c$ is a $2\times2$ matrix. `jax.jacrev` differentiates
the vector-valued solve directly.

In [3]:
J = np.asarray(jax.jacrev(x_star)(c0))
print("d x* / d c :\n", J)

# column-by-column finite-difference check
J_fd = np.zeros((2, 2))
for j in range(2):
    e = np.zeros(2); e[j] = 1e-6
    J_fd[:, j] = (np.asarray(x_star(c0 + e)) - np.asarray(x_star(c0 - e))) / 2e-6
print("finite-diff Jacobian :\n", J_fd)
assert np.allclose(J, J_fd, atol=1e-5)

d x* / d c :
 [[-0.25  0.25]
 [ 0.25 -0.25]]
finite-diff Jacobian :
 [[-0.25  0.25]
 [ 0.25 -0.25]]


## 3. Gradients w.r.t. the matrices $P$ and $G$

OptNet gives gradients w.r.t. **every** datum that enters the optimum — not
just the vectors $c, b, h$ but the matrices $P, G, A$ too ($\nabla P$ is the
symmetric gradient). Here we differentiate the loss w.r.t. a quadratic
penalty matrix $P$ and an inequality matrix $G$. We tighten the first bound
so that inequality is **active** at the optimum — otherwise an inactive
constraint contributes nothing and $\nabla G$ would (correctly) be zero.

In [4]:
G = jnp.array([[1.0, 0.0], [0.0, 1.0]])
h = jnp.array([0.2, 0.8])   # row 0 active at the optimum, row 1 slack

def loss_PG(P, G):
    x = solve_qp(P=P, c=jnp.array([-1.0, -1.2]), G=G, h=h)
    return 0.5 * jnp.sum((x - x_tgt) ** 2)

gP, gG = jax.grad(loss_PG, argnums=(0, 1))(P, G)
print("dloss/dP :\n", np.asarray(gP))
print("dloss/dG :\n", np.asarray(gG), "  <- nonzero: row 0 is active")

# Spot-check entries of both matrix gradients against finite differences.
def perturbed(P00=None, G00=None):
    Pp = P if P00 is None else P.at[0, 0].set(P00)
    Gp = G if G00 is None else G.at[0, 0].set(G00)
    return float(loss_PG(Pp, Gp))

fdP = (perturbed(P00=P[0, 0] + 1e-6) - perturbed(P00=P[0, 0] - 1e-6)) / 2e-6
fdG = (perturbed(G00=G[0, 0] + 1e-6) - perturbed(G00=G[0, 0] - 1e-6)) / 2e-6
print("dloss/dP[0,0]: AD =", float(gP[0, 0]), " FD =", fdP)
print("dloss/dG[0,0]: AD =", float(gG[0, 0]), " FD =", fdG)
assert abs(float(gP[0, 0]) - fdP) < 1e-4
assert abs(float(gG[0, 0]) - fdG) < 1e-4

dloss/dP :
 [[0.      0.0075 ]
 [0.0075  0.04125]]
dloss/dG :
 [[0.0125  0.04375]
 [0.      0.     ]]   <- nonzero: row 0 is active


dloss/dP[0,0]: AD = 2.3393876222629596e-09  FD = 6.033368249447335e-09
dloss/dG[0,0]: AD = 0.012499998185666796  FD = 0.012499989647529741


## 4. `QpLayer`: fixed structure inside a learning loop

`QpLayer` captures `P`/`G`/`A` once and is called with the varying
`c`/`b`/`h`. It composes with `jax.grad`, `jax.jit`, and `vmap`. Here we run
a few steps of gradient descent on `c` so the QP's solution tracks a moving
target — a stand-in for training a QP layer end-to-end.

In [5]:
layer = QpLayer(P=P, A=A)   # equality-constrained QP layer

@jax.jit
def step(c, lr=0.5):
    def L(c):
        return 0.5 * jnp.sum((layer(c, b=b) - x_tgt) ** 2)
    return c - lr * jax.grad(L)(c), L(c)

c = jnp.array([0.0, 0.0])
for k in range(8):
    c, Lk = step(c)
    if k % 2 == 0 or k == 7:
        print(f"step {k}: loss = {float(Lk):.3e}, x* = {np.asarray(layer(c, b=b))}")
print("target:", np.asarray(x_tgt))

step 0: loss = 5.625e-03, x* = [0.36562 0.63437]
step 2: loss = 3.297e-03, x* = [0.35024 0.64976]
step 4: loss = 1.933e-03, x* = [0.33847 0.66153]
step 6: loss = 1.133e-03, x* = [0.32945 0.67055]
step 7: loss = 8.674e-04, x* = [0.32577 0.67423]
target: [0.3 0.7]


### Parallel batch through the layer

`layer.batch(cs)` solves a batch (shape `(B, n)` of linear terms) on the
rayon-parallel path and is differentiable — gradients to the shared `P`/`A`
sum over the batch, gradients to `c` stay per-row.

In [6]:
cs = jnp.array([[-1.0, 0.5], [-0.5, -0.5], [0.2, -1.0]])
xs = layer.batch(cs, b=b)
print("batch solutions:\n", np.asarray(xs))

# differentiable: gradient of the summed batch loss w.r.t. the batched c's
def batch_loss(cs):
    return jnp.sum((layer.batch(cs, b=b) - x_tgt) ** 2)
gcs = jax.grad(batch_loss)(cs)
print("d(batch loss)/d cs:\n", np.asarray(gcs))

batch solutions:
 [[0.75  0.25 ]
 [0.375 0.625]
 [0.075 0.925]]


d(batch loss)/d cs:
 [[-0.45   0.45 ]
 [-0.075  0.075]
 [ 0.225 -0.225]]


## 5. Differentiating an SOCP

`solve_socp` differentiates a second-order cone program — the
complementarity row uses the cone's **arrow operator** in place of the
orthant's diagonal. We use the closed-form ball problem
$x^\*(c) = a - r\,c/\|c\|$ (minimize $c^\top x$ s.t. $\|x-a\|\le r$), whose
Jacobian we know analytically:
$$\frac{\partial x^\*}{\partial c}
= -\frac{r}{\|c\|}\Big(I - \frac{c\,c^\top}{\|c\|^2}\Big).$$

In [7]:
n = 3
a = jnp.array([0.5, -0.2, 0.1])
r_ball = 0.9
G = jnp.vstack([jnp.zeros((1, n)), jnp.eye(n)])   # (n+1) x n

def socp_x(c):
    h = jnp.concatenate([jnp.array([r_ball]), a])
    return solve_socp(P=jnp.zeros((n, n)), c=c, G=G, h=h, cones=[("soc", n + 1)])

c0 = jnp.array([1.0, -2.0, 0.5])
x_cf = np.asarray(a) - r_ball * np.asarray(c0) / np.linalg.norm(np.asarray(c0))
print("x*       :", np.asarray(socp_x(c0)))
print("closed   :", x_cf)

J_ad = np.asarray(jax.jacrev(socp_x)(c0))
cn = np.asarray(c0); nrm = np.linalg.norm(cn)
J_cf = -r_ball / nrm * (np.eye(n) - np.outer(cn, cn) / nrm**2)
print("Jacobian (implicit diff):\n", J_ad)
print("Jacobian (closed form)  :\n", J_cf)
assert np.allclose(J_ad, J_cf, atol=1e-5)

x*       : [ 0.10721  0.58558 -0.0964 ]
closed   : [ 0.10721  0.58558 -0.0964 ]


Jacobian (implicit diff):
 [[-0.31797 -0.14964  0.03741]
 [-0.14964 -0.09352 -0.07482]
 [ 0.03741 -0.07482 -0.37409]]
Jacobian (closed form)  :
 [[-0.31797 -0.14964  0.03741]
 [-0.14964 -0.09352 -0.07482]
 [ 0.03741 -0.07482 -0.37409]]


A scalar SOCP loss, end-to-end through `jax.grad`, checked against finite
differences for good measure.

In [8]:
def socp_loss(c):
    return jnp.sum(socp_x(c) ** 2)

g_ad = np.asarray(jax.grad(socp_loss)(c0))
g_fd = fd_grad(lambda c: float(socp_loss(jnp.asarray(c))), np.asarray(c0), eps=1e-6)
print("grad (implicit diff):", g_ad)
print("grad (finite diff)  :", g_fd)
assert np.allclose(g_ad, g_fd, atol=1e-4)

grad (implicit diff): [-0.25064 -0.12719 -0.00748]
grad (finite diff)  : [-0.25064 -0.12719 -0.00748]


## Recap

- `solve_qp` / `solve_socp` are JAX-differentiable w.r.t. **all** data
  ($P, c, G, h, A, b$) via OptNet implicit differentiation.
- Use `jax.grad` for scalar losses, `jax.jacrev` for the full solution
  Jacobian, and `QpLayer` to embed a fixed-structure problem in a model.
- `layer.batch` / `solve_qp_batch` run rayon-parallel and stay
  differentiable.

See the [Convex Solver chapter](../../docs/src/convex-solver.md) for the
math and the [Acknowledgments](../../docs/src/acknowledgments.md) for the
Clarabel / PaPILO / ECOS / OptNet lineage.